# `viz` plot module demo

Smoke test of every plot function in `viz/` against synthetic data.

The functions are residue-indexed, take numpy arrays, and return
`matplotlib.figure.Figure` objects (so they can be embedded in notebooks,
the Flask UI, or dropped into any other frontend). Synthetic tensors come
from `viz._fakes` and mimic the shapes the extraction layer
(Priyavi/Pranav) is expected to produce -- swap them out once real
extraction lands.

Run all cells; PNGs are written into `viz/examples/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from viz import (
    plot_heatmap,
    plot_heatmap_grid,
    plot_line,
    plot_lines,
    plot_layer_trajectory,
    plot_histogram,
)
from viz._fakes import (
    fake_attention_heads,
    fake_distribution,
    fake_layer_trajectory,
    fake_pair_channel,
    fake_single_channels,
)

## 1. Heatmap (image plot)

Build a synthetic 64x64 attention-style matrix and render it. The diagonal
and a pair of off-diagonal blobs simulate self-attention plus a couple of
long-range contacts. Residues 20 and 45 are highlighted with red gridlines.

In [ ]:
np.random.seed(0)
N = 64
ii, jj = np.indices((N, N))
m = np.exp(-(ii - jj) ** 2 / (2 * 4.0 ** 2))
m += 0.6 * np.exp(-((ii - 20) ** 2 + (jj - 45) ** 2) / (2 * 5.0 ** 2))
m += 0.6 * np.exp(-((ii - 45) ** 2 + (jj - 20) ** 2) / (2 * 5.0 ** 2))
m += 0.05 * np.random.rand(N, N)

fig = plot_heatmap(
    m,
    title='synthetic attention map (N=64)',
    colorbar_label='attention weight',
    highlight_residues=[20, 45],
    save_path='../viz/examples/heatmap_demo.png',
)
fig

## 2. Line plot

A length-64 per-residue signal (e.g. one channel of the single representation
or per-residue attention entropy). Residue 18 is highlighted with a vertical
reference line.

In [ ]:
x = np.arange(N)
s = np.sin(x / 4.0) + 0.4 * np.cos(x / 2.0) + 0.2 * np.random.randn(N)

fig = plot_line(
    s,
    title='synthetic single-representation channel',
    ylabel='channel value',
    highlight_residues=[18],
    save_path='../viz/examples/lineplot_demo.png',
)
fig

## 3. Heatmap grid (per-head subplot)

Render all 8 heads of one attention layer at once. Useful for the "show me
every head of layer 47" UI pattern. Cells share a common color range so
they're visually comparable.

In [ ]:
heads = fake_attention_heads(N=64, H=8, seed=0)
fig = plot_heatmap_grid(
    heads,
    titles=[f'head {i}' for i in range(heads.shape[0])],
    ncols=4,
    suptitle='synthetic attention, 8 heads (N=64)',
    colorbar_label='attention weight',
    save_path='../viz/examples/heatmap_grid_demo.png',
)
fig

## 4. Multi-line overlay

Stack several channels of the single representation on one residue axis.
Good for comparing how a handful of channels behave across the sequence.

In [ ]:
s = fake_single_channels(N=64, C=6, seed=0)
fig = plot_lines(
    s.T,
    labels=[f'channel {c}' for c in range(s.shape[1])],
    title='synthetic single representation, 6 channels',
    ylabel='activation',
    highlight_residues=[18, 45],
    save_path='../viz/examples/lines_demo.png',
)
fig

## 5. Layer trajectory

Track one channel's value across all 48 layers, drawing one line per
selected residue. Reveals "where in the network this signal sharpens."

In [ ]:
traj = fake_layer_trajectory(L=48, N=64, seed=0)
fig = plot_layer_trajectory(
    traj,
    title="one channel's value across 48 layers",
    save_path='../viz/examples/layer_trajectory_demo.png',
)
fig

## 6. Histogram (value distribution)

Quick distributional view of any tensor slice -- here, a synthetic
mixture-of-gaussians sample standing in for a layer's activations.

In [ ]:
dist = fake_distribution(N=20000, seed=0)
fig = plot_histogram(
    dist,
    bins=60,
    title='synthetic activation distribution',
    save_path='../viz/examples/histogram_demo.png',
)
fig

## 7. Shape validation

The plot functions reject wrong-rank input with a clear `ValueError`.

In [ ]:
try:
    plot_heatmap(np.zeros(10))  # 1-D, should fail
except ValueError as e:
    print('plot_heatmap rejected 1-D input:', e)

try:
    plot_line(np.zeros((10, 10)))  # 2-D, should fail
except ValueError as e:
    print('plot_line rejected 2-D input:', e)